In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)


In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.nunique()

In [ ]:
df["Churn"].value_counts()

In [ ]:
df["Churn"].value_counts(normalize=True) * 100

## Clean

In [ ]:
df_clean = df.copy()

In [ ]:
df_clean.duplicated().sum()

In [ ]:
df_clean["customerID"].duplicated().sum()

In [ ]:
df_clean.isnull().sum()

In [ ]:
df_clean[df_clean["TotalCharges"].isnull()]

In [ ]:
df_clean["TotalCharges"].unique()[:20]

In [ ]:
df_clean["TotalCharges"] = (\n    df_clean["TotalCharges"].astype("string").str.strip().replace("", pd.NA)\n)

In [ ]:
df_clean["TotalCharges"] = pd.to_numeric(
    df_clean["TotalCharges"]
)

In [ ]:
df_clean["TotalCharges"].dtype

In [ ]:
df_clean.isnull().sum()

In [ ]:
df_clean[df_clean["TotalCharges"].isnull()][
    ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]
]

In [ ]:
print(df_clean["gender"].unique())
print(df_clean["Contract"].unique())
print(df_clean["InternetService"].unique())
print(df_clean["PaymentMethod"].unique())
print(df_clean["Churn"].unique())

In [ ]:
df_clean["TotalCharges"] = df_clean["TotalCharges"].fillna(0)

In [ ]:
df_clean["TotalCharges"].isnull().sum()

In [ ]:
df_ml = df_clean.drop(columns=["customerID"])

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(PROCESSED_DIR / "telco_churn_clean.csv", index=False)


## who is leaving the company


In [ ]:
df_clean["Churn"].value_counts()

In [ ]:
df_clean["Churn"].value_counts(normalize=True) * 100

In [ ]:
df_clean["Churn"].value_counts().plot(kind="bar")

plt.title("Customer Churn")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.show()

In [ ]:
pd.crosstab(
    df_clean["Contract"],
    df_clean["Churn"],
    normalize="index"
) * 100

In [ ]:
contract_churn = pd.crosstab(
    df_clean["Contract"],
    df_clean["Churn"],
    normalize="index"
) * 100

contract_churn.plot(
    kind="bar",
    stacked=True
)

plt.title("Churn Rate by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Percentage")
plt.xticks(rotation=0)
plt.legend(title="Churn")
plt.show()

In [ ]:
df_clean["tenure"].describe()

In [ ]:
df_clean["tenure_group"] = pd.cut(
    df_clean["tenure"],
    bins=[0, 6, 12, 24, 48, 72],
    labels=[
        "0-6 months",
        "7-12 months",
        "13-24 months",
        "25-48 months",
        "49-72 months"
    ]
)

In [ ]:
pd.crosstab(
    df_clean["tenure_group"],
    df_clean["Churn"],
    normalize="index"
) * 100

In [ ]:
df_clean.groupby("Churn")["MonthlyCharges"].mean()

In [ ]:
df_clean.groupby("Churn")["MonthlyCharges"].mean().plot(
    kind="bar"
)

plt.title("Average Monthly Charges by Churn")
plt.xlabel("Churn")
plt.ylabel("Average Monthly Charges")
plt.xticks(rotation=0)
plt.show()

## Feature Engineering

In [ ]:
df_clean.dtypes

In [ ]:
df_ml = df_clean.drop(
    columns=["customerID", "tenure_group"]
).copy()

In [ ]:
target = df_ml["Churn"].map({
    "No": 0,
    "Yes": 1
})
if target.isna().any():
    invalid_labels = sorted(df_ml.loc[target.isna(), "Churn"].dropna().astype(str).unique())
    raise ValueError(f"Unexpected or missing Churn labels: {invalid_labels}")
df_ml["Churn"] = target.astype("int8")

X = df_ml.drop(columns=["Churn"])
y = df_ml["Churn"]

print("Features:", X.shape)
print("Target distribution:")
print(y.value_counts(normalize=True).rename("proportion"))


## Train-test split

Split the data before fitting any learned preprocessing steps. Stratification preserves the churn/non-churn proportions in both sets.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## Preprocessing and model training

The preprocessing steps are kept inside one Scikit-learn pipeline. Imputers and the encoder are fitted only on the training data, then applied to the test data. This avoids preprocessing leakage.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

categorical_cols = X_train.select_dtypes(include=["object"]).columns
numerical_cols = X_train.select_dtypes(include=["number"]).columns

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numerical_cols),
    ("cat", categorical_transformer, categorical_cols)
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

## Model evaluation

Evaluate the held-out test set using multiple classification metrics. Churn recall is especially useful because it measures how many actual churners the model identifies.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

y_pred = model.predict(X_test)

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred, zero_division=0):.3f}")
print(f"F1 score:  {f1_score(y_test, y_pred, zero_division=0):.3f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("Customer Churn - Confusion Matrix")
plt.show()

## Save the complete pipeline

Save the fitted pipeline, not just the classifier. It includes the preprocessing steps required to make predictions on new customer records.

In [ ]:
import joblib

model_path = MODEL_DIR / "churn_logistic_model.pkl"
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, model_path)
print(f"Saved model to: {model_path}")
